# Step-1 : Install  the required libraries/packages

In [ ]:
# pip install pandas
# pip install openpyxl


# Step-2 : Import the required libraries/packages

In [1]:
import pandas as pd
import os

# Step 3: Load/Import data & Build connection

In [6]:
df = pd.read_excel("input_files/customer_sample_500.csv.xlsx", dtype=str)

# Replace NaN with empty string
df = df.fillna("")

print("File loaded successfully")

File loaded successfully


# Step 4: Data Quality | Completeness Check

## Datasets using: 
- customer_sample_500


In [18]:
def completeness_check(df, output_file="QualityCheck_results.xlsx"):
    """
    Perform a completeness check on each column of the DataFrame and
    store the results in an Excel sheet.

    This function evaluates missing or empty values across all columns
    in the given DataFrame. A value is considered missing if it is either:
    - an empty string
    - a string containing only whitespace

    The results are stored in a structured format and written to an Excel
    file located in the 'output_files' folder. If the file already exists,
    the 'Completeness_Check' sheet will be replaced to ensure consistent
    output across multiple executions.

    Dictionary Usage
    ----------------
    Each row of the result is constructed using a dictionary:
        Key   → Column name / metric name
        Value → Computed result

    Example:
        {
            "Column": "SIRET",
            "Missing Values": 5
        }

    Parameters
    ----------
    df : pandas.DataFrame
        Input DataFrame on which completeness checks are performed.

    output_file : str, optional
        Name of the Excel file where results will be stored.
        Default is "QualityCheck_results.xlsx".

    Returns
    -------
    None
        The function writes results directly to an Excel file.
    """

    results = []

    # Loop through each column in the DataFrame
    for col in df.columns:

        # Count missing or empty values
        # Convert values to string → remove whitespace → check empty
        missing = df[col].astype(str).str.strip().eq("").sum()

        # Store result using dictionary (key → value format)
        results.append({
            "Column": col,                # Key: column name
            "Missing Values": missing     # Value: count of missing entries
        })

    # Convert list of dictionaries into a DataFrame
    result_df = pd.DataFrame(results)

    # Define output folder path
    folder_path = "output_files"

    # Construct full file path
    file_path = os.path.join(folder_path, output_file)

    # Write results to Excel
    # If file exists → replace the same sheet
    # If not → create new file
    with pd.ExcelWriter(
        file_path,
        engine="openpyxl",
        mode="a" if os.path.exists(file_path) else "w",
        if_sheet_exists="replace"
    ) as writer:

        # Write DataFrame to Excel sheet
        result_df.to_excel(writer, sheet_name="Completeness_Check", index=False)


# Run completeness check
completeness_check(df)

# Step -5: Data Quality | Uniquenss check
## Datasets using:
- customer_sample_500

In [19]:
#--uniqueness_check-----

# ---- Step 1: Find columns that contain unique values ----
def find_unique_columns(df):
    """
    Identify columns where all values are unique.

    A column is considered unique if the number of distinct values
    (including nulls) is equal to the total number of rows.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataset.

    Returns
    -------
    list
        List of column names that contain only unique values.
    """

    unique_cols = []

    # Loop through each column
    for col in df.columns:

        # Check if all values in the column are unique
        if df[col].nunique(dropna=False) == len(df):
            unique_cols.append(col)

    return unique_cols


# ---- Step 2: Perform uniqueness check ----
def uniqueness_check(df, output_file="QualityCheck_results.xlsx"):
    """
    Perform uniqueness validation on identified unique columns.

    This function:
    1. Detects columns that contain only unique values.
    2. Checks for duplicate occurrences in those columns.
    3. Stores results in an Excel file under two sheets:
       - "Uniqueness_Columns"
       - "Uniqueness_Check"

    Dictionary Usage
    ----------------
    Results are stored using dictionary (key–value) format:

        {
            "Column": column_name,
            "Duplicate_Count": count
        }

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataset for validation.

    output_file : str, optional
        Excel file name where results will be stored.

    Returns
    -------
    None
        Results are written directly to an Excel file.
    """

    # Step 1: Get columns with unique values
    unique_cols = find_unique_columns(df)

    # Convert list of unique columns to DataFrame
    unique_df = pd.DataFrame(unique_cols, columns=["Unique_Columns"])

    duplicate_results = []

    # Step 2: Check duplicates in unique columns
    for col in unique_cols:

        # Count how many values appear more than once
        duplicates = (df[col].value_counts() > 1).sum()

        # Store results using dictionary (key → value)
        duplicate_results.append({
            "Column": col,               # Key: column name
            "Duplicate_Count": duplicates  # Value: duplicate count
        })

    # Convert results into DataFrame
    duplicate_df = pd.DataFrame(duplicate_results)

    # Define output folder and file path
    folder_path = "output_files"
    file_path = os.path.join(folder_path, output_file)

    # Write results to Excel
    # If file exists → replace sheets
    # If not → create new file
    with pd.ExcelWriter(
        file_path,
        engine="openpyxl",
        mode="a" if os.path.exists(file_path) else "w",
        if_sheet_exists="replace"
    ) as writer:

        # Sheet 1: list of unique columns
        unique_df.to_excel(writer, sheet_name="Uniqueness_Columns", index=False)

        # Sheet 2: duplicate counts
        duplicate_df.to_excel(writer, sheet_name="Uniqueness_Check", index=False)


# Run uniqueness check
uniqueness_check(df)

# Step-6 : Quality check | Validity check
## Datasets using :
- customer_sample_500 

In [20]:
def run_validity_checks(df, output_file="QualityCheck_results.xlsx"):
    """
    Perform validity checks on specific columns of the DataFrame.

    This function validates data based on predefined rules such as:
    - Length checks (e.g., SIRET, SIREN, COUNTRY_CD)
    - Numeric format checks (postal codes, GERS)
    - Allowed value checks (delivery block fields)

    Additional rules are dynamically added for delivery-related columns.

    Dictionary Usage
    ----------------
    Validation rules are stored in a dictionary where:
        Key   → Name of the validation rule
        Value → Boolean condition identifying invalid records

    Example:
        {
            "Invalid_SIRET": df["SIRET"].str.len() != 14
        }

    Results are also stored using dictionary format:
        {
            "Check": rule_name,
            "Invalid_Count": count
        }

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataset to validate.

    output_file : str, optional
        Excel file name where results will be stored.

    Returns
    -------
    None
        Results are written directly to an Excel file.
    """

    # Dictionary defining validation rules (key → rule name, value → condition)
    validity_rules = {
        "Invalid_SIRET": (df["SIRET"].str.len() != 14),

        "Invalid_SIREN": (df["SIREN"].str.len() != 9),

        "Invalid_Country_Code": (df["COUNTRY_CD"].str.len() != 2),

        "Invalid_SAP_Postal_Code": (~df["SAP_POSTAL_ZIP_CD"].str.match(r'^\d+$')),

        "Invalid_LUCI_Postal_Code": (~df["LUCI_POSTAL_ZIP_CD"].str.match(r'^\d+$')),

        "Invalid_Customer_Delivery_Block":
            (~df["CUSTOMER_DELIVERY_BLOC"].isin(["00", "01"])),

        "Invalid_GERS":
            (~df["GERS"].str.match(r'^\d+$'))
    }

    # List of columns requiring the same validation rule
    delivery_columns = [
        "CUSTOMER_ORDER_BLOCK",
        "CUSTOMER_SALES_DELIVERY_BLOC",
        "CUSTOMER_SALES_ORDER_BLOCK",
        "DELETION_BLOCK"
    ]

    # Dynamically add validation rules for delivery-related columns
    for col in delivery_columns:
        validity_rules[f"{col}_Invalid"] = ~df[col].isin(["00", "01"])

    results = []

    # Evaluate each rule and count invalid records
    for rule, condition in validity_rules.items():

        invalid_count = condition.sum()

        # Store results using dictionary (key → value)
        results.append({
            "Check": rule,              # Key: rule name
            "Invalid_Count": invalid_count  # Value: number of invalid records
        })

    # Convert results list into DataFrame
    validity_df = pd.DataFrame(results)

    # Define output file path
    folder_path = "output_files"
    file_path = os.path.join(folder_path, output_file)

    # Write results to Excel (overwrite same sheet every run)
    with pd.ExcelWriter(
        file_path,
        engine="openpyxl",
        mode="a" if os.path.exists(file_path) else "w",
        if_sheet_exists="replace"
    ) as writer:

        validity_df.to_excel(writer, sheet_name="Validity_Check", index=False)


# Run validity checks
run_validity_checks(df)

# Step-7: Quality Checks | Consistency check
## Data sets using :
- customer_sample_500

In [21]:
# Consistency check 

def run_consistency_checks(df, output_file="QualityCheck_results.xlsx"):
    """
    Perform consistency validation between related columns in the DataFrame.

    This function checks whether values in related columns are consistent.
    Currently, it verifies whether COUNTRY_CD and LUCI_COUNTRY_CD match
    for each record.

    Dictionary Usage
    ----------------
    Consistency rules are stored in a dictionary where:
        Key   → Name of the consistency check
        Value → Boolean condition identifying mismatched records

    Example:
        {
            "Country_Mismatch": df["COUNTRY_CD"] != df["LUCI_COUNTRY_CD"]
        }

    Results are also stored using dictionary format:
        {
            "Check": rule_name,
            "Mismatch_Count": count
        }

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataset to validate.

    output_file : str, optional
        Excel file name where results will be stored.

    Returns
    -------
    None
        Results are written directly to an Excel file.
    """

    # Dictionary defining consistency rules (key → rule name, value → condition)
    consistency_rules = {
        "Country_Mismatch": df["COUNTRY_CD"] != df["LUCI_COUNTRY_CD"]
    }

    results = []

    # Loop through each rule and count mismatches
    for rule, condition in consistency_rules.items():

        mismatch_count = condition.sum()

        # Store results using dictionary (key → value format)
        results.append({
            "Check": rule,                 # Key: rule name
            "Mismatch_Count": mismatch_count  # Value: number of mismatches
        })

    # Convert results list into a DataFrame
    consistency_df = pd.DataFrame(results)

    # Define output file path
    folder_path = "output_files"
    file_path = os.path.join(folder_path, output_file)

    # Write results to Excel (overwrite same sheet every run)
    with pd.ExcelWriter(
        file_path,
        engine="openpyxl",
        mode="a" if os.path.exists(file_path) else "w",
        if_sheet_exists="replace"
    ) as writer:

        consistency_df.to_excel(writer, sheet_name="Consistency_Check", index=False)


# Run consistency check
run_consistency_checks(df)